# 00-12 Institutional Quantitative Research Framework
## Professional Asset Management Quality – No Lookahead Bias, No Survivorship Bias, No OOS Optimization

This notebook is **pure orchestration** – no business logic inside. All logic lives in `src/quant_framework/` as classes / pure functions with type hints, docstrings, validation, deterministic output.

**Architecture:**
- 00 Config – immutable ResearchConfig
- 01 Data – DataManager (yfinance, cache, timezone UTC, corporate actions auto_adjust=True, coverage/missing/history reports, reproducible)
- 02 Universe – UniverseManager + AssetMetadataEngine (sector, macro bucket, risk bucket, automatic classification XLU=Utilities Defensive Low etc)
- 03 Feature Engineering – FeatureStore lazy evaluated, each feature DataFrame, registry with economic justification
- 04 Signal Engine – TEMACD, BTC_TEMACD, LT_MA_CROSS etc, modular, testable
- 05 Regime Engine – plugin system (XLUCanary, QQQCanary, CreditSpreadCanary etc) each returns signal/confidence/probability/historical_accuracy/turnover, enablable/disableable + ExpertSystem declarative knowledge base
- 06 Portfolio Construction – Equal Weight, Inverse Vol, Risk Parity, Max Div, Min Var, Vol Targeting, Kelly, Dynamic Sizing, Cash Allocation – separated from signals
- 07 Execution Engine – realistic simulator with transaction costs, commissions, slippage, partial fills, rebalance frequency, cash drag, fractional, min lot, execution delay, next open/close, no lookahead bias
- 08 Performance Analytics – Sharpe, Sortino, Calmar, MaxDD, WinRate, PF, PSR, DSR
- 09 Optimization – grid search only on train, never OOS
- 10 Walk Forward Validation – train/test split, no overlap, no OOS optimization
- 11 Stress Testing – historical (2008, 2020, 2022) + synthetic vol/rate shock
- 12 Reporting – Reporter embeds PNG as HTML, JSON, Markdown, deterministic
- Risk Engine – Portfolio Vol, MRC, CRC, VaR, CVaR, Ulcer, Tail Ratio, Skew, Kurtosis, Rolling Beta/Correlation, Risk Concentration, Risk Budget
- Robustness – Monte Carlo, Bootstrap, Noise Injection, Param Perturbation, Randomized Start, Missing Data, Cost/Slippage Sensitivity, robustness score
- Experiment Registry – ID, Timestamp, Git hash, Universe, Date range, Train/Validation/Test, Feature list, Signal list, Portfolio model, Execution model, Costs, Optimization, Metrics, Seed, Notebook version, re-runnable JSON


In [ ]:
# 00 Config – immutable, validated, no business logic
from pathlib import Path
import sys
sys.path.insert(0, '../src')
from quant_framework.config.settings import ResearchConfig, DataConfig, UniverseConfig, ExecutionConfig, PortfolioConfig, RegimeConfig, PerformanceConfig

config = ResearchConfig(
    data=DataConfig(cache_dir=Path("../data/cache"), auto_adjust=True, timezone="UTC"),
    universe=UniverseConfig(name="quant_rick_30", benchmark="SPY"),
    execution=ExecutionConfig(commission_pct=0.0005, slippage_bps=2.0, execution_delay=1, execution_price="next_open", initial_capital=100000.0),
    portfolio=PortfolioConfig(method="equal_weight", target_vol=0.20, cash_allocation=0.0, max_position_weight=0.40),
    regime=RegimeConfig(enabled_plugins=["XLUCanary","QQQCanary","CreditSpreadCanary","ConsumerCanary","VIXCanary","YieldCurveCanary","DollarCanary","OilCanary","GoldCanary"]),
    performance=PerformanceConfig(benchmark="SPY"),
    random_seed=42,
    project_name="TQQQ_TEMACD_Institutional"
)
print("Config:", config.to_dict())


In [ ]:
# 01 Data – DataManager institutional level
from quant_framework.data.data_manager import DataManager
from quant_framework.data.asset_metadata import AssetMetadataEngine

dm = DataManager(cache_dir=config.data.cache_dir, auto_adjust=config.data.auto_adjust, timezone=config.data.timezone)
tickers = config.universe.tickers
print(f"Validated tickers: {tickers[:10]}... total {len(tickers)}")
data = dm.download(tickers=tickers, start="2020-01-01", end="2025-07-15", interval="1d", use_cache=True)
print(f"Downloaded {len(data)} tickers")
coverage = dm.get_coverage_report()
print(coverage.head())


In [ ]:
# Asset Metadata Engine – automatic classification
metadata_engine = AssetMetadataEngine(cache_data=data)
for t in ["XLU","QQQ","GLD","DBMF","TQQQ","SPY"]:
    try:
        m = metadata_engine.get_metadata(t)
        print(f"{t}: asset_class={m.asset_class}, sector={m.sector}, macro={m.macro_bucket}, risk={m.risk_bucket}, beta={m.beta}")
    except Exception as e:
        print(t, e)


In [ ]:
# 02 Universe – independent
from quant_framework.universe.universe_manager import UniverseManager
um = UniverseManager(config=config.universe, metadata_engine=metadata_engine)
universe = um.get_universe()
print(f"Universe {universe.name}: {len(universe.tickers)} tickers")
print(f"Defensive: {um.filter_by_macro('Defensive')}")
print(f"Risk On: {um.filter_by_macro('Risk On')}")


In [ ]:
# 03 Feature Engineering – lazy evaluated
from quant_framework.features.feature_store import FeatureStore, FeatureRegistry
fs = FeatureStore(price_data=data, benchmark_ticker=config.universe.benchmark)
mom = fs.get_feature("TQQQ", FeatureRegistry.MOMENTUM, window=252)
rsi = fs.get_feature("TQQQ", FeatureRegistry.RSI, window=14)
print(f"Momentum last: {mom.iloc[-1].values[0]:.3f}, RSI last: {rsi.iloc[-1].values[0]:.1f}")
print(f"Cache stats: {fs.cache_stats()}")
cs_rank = fs.get_cross_sectional_rank(feature=FeatureRegistry.MOMENTUM, window=252)
print(f"Cross-sectional rank shape: {cs_rank.shape}")


In [ ]:
# 04 Signal Engine – modular, no portfolio logic
from quant_framework.signals.signal_engine import SignalEngine, TEMACDSignal
se = SignalEngine(feature_store=fs)
se.register_signal(TEMACDSignal(len1=5, len2=50, len3=222, fast=61, slow=70, sig=15))
signals = se.generate_all(tickers=["TQQQ","SPY","QQQ"])
print(signals["TQQQ"]["TQQQ_TEMACD"].tail())


In [ ]:
# 05 Regime Engine – plugin system + Expert System declarative
from quant_framework.regime.plugins import XLUCanary, QQQCanary, CreditSpreadCanary, VIXCanary
from quant_framework.regime.regime_engine import RegimeEngine
from quant_framework.regime.expert_system import default_knowledge_base, ExpertSystem
plugins = [XLUCanary(), QQQCanary(), CreditSpreadCanary(), VIXCanary()]
re = RegimeEngine(config=config.regime, plugins=plugins)
regime_result = re.evaluate(data)
print(f"Regime: {regime_result.regime}, conf {regime_result.confidence:.2f}")
kb = default_knowledge_base()
es = ExpertSystem(knowledge_base=kb)
expert_result = es.evaluate(regime_result.contributing_canaries)
print(expert_result)


In [ ]:
# 06 Portfolio Construction – separated from signals
from quant_framework.portfolio.construction import PortfolioConstructor
import pandas as pd
pc = PortfolioConstructor(config=config.portfolio)
ew = pc.equal_weight(["TQQQ","QQQ","SPY","GLD","IEF"])
print(ew.weights)


In [ ]:
# 07 Execution Engine – no lookahead bias
from quant_framework.execution.simulator import ExecutionSimulator, ExecutionConfig, Order
exec_config = ExecutionConfig(execution_delay=1, execution_price="next_open")
sim = ExecutionSimulator(config=exec_config)
print("ExecutionSimulator ready – no lookahead, next bar execution")


In [ ]:
# 08 Performance Analytics – PSR/DSR
from quant_framework.performance.analytics import PerformanceAnalytics
pa = PerformanceAnalytics()
print("PerformanceAnalytics ready")


In [ ]:
# 09-11 Optimization, Walk Forward, Stress Testing, Risk Engine, Robustness, Reporting, Experiment Registry
from quant_framework.risk.risk_engine import RiskEngine
from quant_framework.validation.robustness import RobustnessFramework
from quant_framework.experiment.registry import ExperimentRegistry
from quant_framework.reporting.reporting_engine import ReportingEngine
print("RiskEngine, RobustnessFramework, ExperimentRegistry, ReportingEngine ready")
risk_engine = RiskEngine()
robust = RobustnessFramework(random_seed=42, n_simulations=50)
registry = ExperimentRegistry(registry_dir="../reports/experiments_registry")
reporter = ReportingEngine(output_dir="../reports/full")
print("All engines instantiated – institutional grade")
